<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/06_pandas_intro/06_3_Selecting_Filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06.3 — Selecting, Filtering, and Slicing

## Learning Objectives

By the end of this notebook you should be able to:

- Pull a specific row or cell using `.loc[]` (label-based) and `.iloc[]` (position-based)
- Filter rows down to a subset using boolean masks
- Combine multiple conditions with `&`, `|`, `.isin()`, and `~`
- Write compact, readable filters with `.query()`
- Understand the difference between a copy and a view, and avoid `SettingWithCopyWarning`

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
df = pd.read_csv(url)
df.columns = df.columns.str.lower()
df = df.rename(columns={
    'siblings/spouses aboard': 'sibsp',
    'parents/children aboard': 'parch'
})
df.head()

## The Problem This Notebook Solves

The Titanic dataset has 887 rows. Almost no real analysis works on all rows at once.

You'll constantly want to zoom in on a specific group:
- *"Show me only the passengers who survived."*
- *"Show me first-class passengers older than 40."*
- *"Show me the record for this specific passenger."*
- *"Show me just the name, age, and survival columns."*

This notebook gives you the complete toolkit for extracting exactly the subset of rows and columns you need.

## 1. Looking Up a Specific Row

Sometimes you want a single record — maybe to spot-check a value or investigate an outlier. Two tools do this:

- **`.loc[label]`** — selects by the index label (here, the row number).
- **`.iloc[position]`** — selects by integer position, 0-based, like a Python list.

When the index is just 0, 1, 2, … (as it is here), both give the same result. The difference matters when you've filtered or sorted the DataFrame and the index no longer matches position.

In [ ]:
# Look up a single row by its label
df.loc[0]

In [ ]:
# Look up specific columns for a specific row
df.loc[0, ["name", "age", "pclass", "survived"]]

In [ ]:
# .iloc[] uses position, not label
# After filtering, row 0 may not be at position 0 — iloc always gives the nth row
print("First row via iloc:", df.iloc[0]["name"])
print("Last row via iloc: ", df.iloc[-1]["name"])

### Selecting a Range of Rows

Both `.loc[]` and `.iloc[]` support slicing, but with one important difference:
- `.loc[]` slice end is **inclusive** (the row with that label is included).
- `.iloc[]` slice end is **exclusive** (like standard Python slicing).

In [ ]:
# .iloc[]: positions 0 through 4 — gives 5 rows (end exclusive)
df.iloc[0:5][["name", "age", "survived"]]

In [ ]:
# Selecting specific rows AND specific columns simultaneously
df.loc[[0, 5, 10], ["name", "pclass", "survived"]]

## 2. Filtering Rows: Boolean Masks

Single-row lookups are useful for spot-checking, but most analyses focus on a group defined by a condition. To do that, you write a comparison expression on a column. The result is a **boolean Series** — a column of `True`/`False` values telling you which rows match.

Then you pass that boolean Series back into `.loc[]`, and pandas keeps only the `True` rows.

In [ ]:
# Who survived?
survived_mask = df["survived"] == 1
# This is 887 True/False values — one per passenger
survived_mask.head(10)

In [ ]:
survivors = df.loc[survived_mask]
print(f"{len(survivors)} of {len(df)} passengers survived ({len(survivors)/len(df):.0%})")
survivors[["name", "sex", "pclass", "age", "fare"]].head()

In [ ]:
# You'll almost always write the mask inline, without naming it separately
first_class = df.loc[df["pclass"] == 1]
print(f"First-class passengers: {len(first_class)}")
first_class[["name", "pclass", "fare"]].head()

## 3. Combining Conditions

Real questions usually involve more than one condition. Use `&` (AND) and `|` (OR) to combine boolean masks.

**Always put each condition in its own parentheses.** Python's operator precedence evaluates `&` before `==`, so without parentheses the expression silently does the wrong thing.

In [ ]:
# First-class passengers who survived
first_class_survivors = df.loc[
    (df["pclass"] == 1) & (df["survived"] == 1)
]
print(f"First-class survivors: {len(first_class_survivors)}")
first_class_survivors[["name", "sex", "age", "fare"]].head()

In [ ]:
# Women who were in first OR second class
upper_class_women = df.loc[
    (df["sex"] == "female") & (df["pclass"].isin([1, 2]))
]
print(f"Women in 1st or 2nd class: {len(upper_class_women)}")
print(f"Of those, survived: {upper_class_women['survived'].sum()}")

### `.isin()` — Clean Multi-Value Matching

When you want to check whether a value belongs to a set of options, `.isin([...])` is cleaner than chaining multiple `|` conditions.

In [ ]:
# Equivalent to (pclass == 1) | (pclass == 2), but easier to read
df.loc[df["pclass"].isin([1, 2])][["name", "pclass"]].head()

### `~` — Negation (NOT)

Prefix a mask with `~` to invert it — keep only the rows where the condition is `False`.

In [ ]:
# Passengers who did NOT survive
did_not_survive = df.loc[~(df["survived"] == 1)]
# Equivalently: df.loc[df["survived"] == 0]
print(f"Did not survive: {len(did_not_survive)}")

## 4. `.query()` — More Readable Filters

For simple conditions, `.query()` accepts a plain-English string and often reads more naturally than a boolean mask.

In [ ]:
# Same as df.loc[(df["pclass"] == 1) & (df["age"] > 40)]
older_first_class = df.query("pclass == 1 and age > 40")
older_first_class[["name", "pclass", "age", "fare"]].head()

In [ ]:
# 'in' works naturally inside .query()
df.query("pclass in [1, 2] and survived == 1")[["name", "pclass"]].head()

`.query()` has limits: it doesn't work well with column names containing spaces or special characters, and it can't reference complex computed values. Use boolean masks when you need full flexibility — but reach for `.query()` when the condition is simple and you want it to be easy to read.

## 5. Copy vs View — A Practical Warning

When you filter a DataFrame, pandas sometimes returns a **view** (a live window into the original data) and sometimes a **copy** (completely independent). The distinction matters the moment you try to *modify* the result.

If you modify a view, you may accidentally modify the original. If you modify a copy, your change may silently disappear. pandas warns you about this ambiguity with `SettingWithCopyWarning`.

In [ ]:
# This may raise SettingWithCopyWarning
survivors = df[df["survived"] == 1]
survivors["fare"] = 0   # Are we changing a copy or the original?

In [ ]:
# The fix: call .copy() immediately after any filter you plan to modify
survivors = df[df["survived"] == 1].copy()
survivors["fare"] = 0   # Clearly modifying an independent copy — no warning

# Confirm the original is unchanged
print("Original fare for row 1:", df.loc[1, "fare"])

**Rule:** if you filter and plan to *read* the result, `.copy()` is unnecessary. If you filter and plan to *add or change columns*, always call `.copy()` first.

## 6. Putting It Together: Answering a Real Question

Let's put these tools to work on a question that requires combining several of them:

*"Among third-class women, who survived and who didn't — and what were they paying?"*


In [ ]:
third_class_women = df.loc[
    (df["pclass"] == 3) & (df["sex"] == "female")
].copy()

survived = third_class_women.loc[third_class_women["survived"] == 1]
did_not  = third_class_women.loc[third_class_women["survived"] == 0]

print(f"Third-class women:         {len(third_class_women)}")
print(f"  Survived:                {len(survived)} ({len(survived)/len(third_class_women):.0%})")
print(f"  Did not survive:         {len(did_not)} ({len(did_not)/len(third_class_women):.0%})")
print()
print(f"Average fare — survivors:     £{survived['fare'].mean():.2f}")
print(f"Average fare — did not survive: £{did_not['fare'].mean():.2f}")

## Your Turn

1. How many passengers were children (age < 18)? Of those, how many survived?
2. Find passengers who paid more than £100. Who are they? (Show name, class, fare, survived.) Were most of them from first class?
3. Find every passenger whose name contains `"Miss."`. How many are there? What fraction survived?
   *Hint: `.str.contains()`*
4. Select male passengers in first class using `.query()`. Then do the same with a boolean mask. Confirm both return the same number of rows.
5. Using `.iloc[]`, select rows at positions 200 through 209 and show only `name`, `age`, `pclass`. Does `.loc[200:209]` give the same result? Why?

In [ ]:
# Your code here